# CURE-Rec — run all remaining reviewer actions

Run this notebook top-to-bottom. No YAML configuration is changed. The CRN study uses an internal stochastic click-feedback variant because the archived baseline has zero click feedback and therefore cannot expose random-shock variance. Phase A is not rerun.


In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd

CANDIDATES=[Path.cwd(),Path.cwd()/'paper-ideas'/'CURE-Rec'/'code',*Path.cwd().parents]
ROOT=next(p for p in CANDIDATES if (p/'pyproject.toml').exists() and (p/'cure_rec').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from cure_rec.config import load_settings
from cure_rec.pipeline import run_experiment
from cure_rec.revision_suite import paired_user_statistics

CONFIG=ROOT/'configs'/'curesim_full.yaml'
RESULTS=ROOT/'results'/'reviewer_phase_assets'
RUN_ALL=True
CRN_SEEDS=tuple(range(300,320))
CRN_CLICK_FEEDBACK_WEIGHT=0.35
print('Root:',ROOT)
print('Registered YAML is read-only; CRN override:',CRN_CLICK_FEEDBACK_WEIGHT)

Root: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code
Registered YAML is read-only; CRN override: 0.35


## 1. Inspect archived Phase B/C assets


In [2]:
for name in ['phase_b_objective_constraint_ablation.csv','phase_c_sampled_shapley_fidelity.csv']:
    p=RESULTS/'tables'/name
    if p.exists():
        print('\n'+name)
        display(pd.read_csv(p))
    else: print('Missing:',p)


phase_b_objective_constraint_ablation.csv


,objective,constraint_mode,penalty,mask,interventions,feasible,runtime_seconds,cost,relevance_delta_lower,provider_disparity_upper,fatigue_upper
0,maximin,hard,0.00,1,repeat_cap,True,0.000630,0.05,-0.041478,0.244275,0.0
1,maximin,penalty,0.00,1,repeat_cap,True,0.000642,0.05,-0.041478,0.244275,0.0
2,maximin,penalty,0.25,1,repeat_cap,True,0.000623,0.05,-0.041478,0.244275,0.0
3,maximin,penalty,0.50,1,repeat_cap,True,0.000622,0.05,-0.041478,0.244275,0.0
4,maximin,penalty,1.00,1,repeat_cap,True,0.000626,0.05,-0.041478,0.244275,0.0
5,maximin,penalty,2.00,1,repeat_cap,True,0.000669,0.05,-0.041478,0.244275,0.0
6,maximin,penalty,5.00,1,repeat_cap,True,0.000642,0.05,-0.041478,0.244275,0.0
7,maximin,penalty,10.00,1,repeat_cap,True,0.000617,0.05,-0.041478,0.244275,0.0
8,mean,hard,0.00,1,repeat_cap,True,0.000866,0.05,-0.041478,0.244275,0.0
9,mean,penalty,0.00,1,repeat_cap,True,0.000879,0.05,-0.041478,0.244275,0.0



phase_c_sampled_shapley_fidelity.csv


,budget,mae,max_error,sign_agreement,rank_correlation,runtime_seconds
0,32,0.000849,0.001590,1.0,1.0,0.000713
1,128,0.001550,0.003293,1.0,1.0,0.000657
2,512,0.000502,0.000822,1.0,1.0,0.002121
3,2048,0.000207,0.000377,1.0,1.0,0.007253


## 2. Simulator-backed CRN paired-difference study

Pairs are empty→repeat_cap (0→1) and repeat_cap→repeat_cap+tail_slot (1→5). The only CRN-specific change is the internal click-feedback variant; the registered YAML is not edited.

In [3]:
def pair_value(settings, seed, a, b, common):
    cfg=settings.model_copy(deep=True)
    cfg.run.seed=int(seed)
    cfg.run.common_random_numbers=bool(common)
    cfg.simulator.click_feedback_weight=CRN_CLICK_FEEDBACK_WEIGHT
    cfg.run.name=f'crn-click-{common}-{seed}-{a}-{b}'
    cfg.run.output_root=ROOT/'runs'/'reviewer-crn-click'
    logger,game,_=run_experiment(cfg)
    diffs=[]
    for scenario in game.scenario_games.values():
        diffs.append(float(scenario.values[b].utility-scenario.values[a].utility))
    return float(np.mean(diffs)),logger.run_dir

def run_crn():
    settings=load_settings(CONFIG)
    rows=[]
    for a,b in ((0,1),(1,5)):
        for seed in CRN_SEEDS:
            crn,crn_dir=pair_value(settings,seed,a,b,True)
            iid,iid_dir=pair_value(settings,seed,a,b,False)
            rows.append({'seed':seed,'mask_a':a,'mask_b':b,'crn_difference':crn,'independent_difference':iid,'paired_difference':crn-iid,'crn_run':Path(crn_dir).relative_to(ROOT).as_posix(),'independent_run':Path(iid_dir).relative_to(ROOT).as_posix()})
    frame=pd.DataFrame(rows)
    summary=frame.groupby(['mask_a','mask_b']).agg(crn_variance=('crn_difference','var'),independent_variance=('independent_difference','var'),crn_mean=('crn_difference','mean'),independent_mean=('independent_difference','mean'),n=('seed','count')).reset_index()
    summary['variance_ratio']=summary['crn_variance']/summary['independent_variance']
    out=RESULTS/'crn_click_feedback'
    out.mkdir(parents=True,exist_ok=True)
    frame.to_csv(out/'crn_paired_differences.csv',index=False)
    summary.to_csv(out/'crn_summary.csv',index=False)
    (out/'crn_manifest.json').write_text(json.dumps({'seeds':list(CRN_SEEDS),'pairs':[[0,1],[1,5]],'click_feedback_weight':CRN_CLICK_FEEDBACK_WEIGHT,'registered_config_changed':False,'claim_scope':'CURE-Sim stochastic click-feedback CRN diagnostic'},indent=2))
    return frame,summary

if RUN_ALL:
    crn_rows,crn_summary=run_crn()
    display(crn_summary)

2026-08-15 23:53:38,603 | INFO | run_started | {"config_hash": "cc9e64e811b3562b", "run_id": "crn-click-True-300-0-1-20260815T225338Z-4383fec9"}
2026-08-15 23:53:38,604 | INFO | exact_game_started | {}
2026-08-15 23:53:38,606 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "nominal"}
2026-08-15 23:56:46,480 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.14078479831440316, "scenario": "nominal", "shapley_efficiency_gap": 0.0}
2026-08-15 23:56:46,481 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "fatigue_stress"}
2026-08-16 00:00:00,409 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.12829884329851904, "scenario": "fatigue_stress", "shapley_efficiency_gap": 0.0}
2026-08-16 00:00:00,410 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "popularity_stress"}
2026-08-16 00:02:56,113 | INFO | scenario_game_completed | {"grand_coaliti

,mask_a,mask_b,crn_variance,independent_variance,crn_mean,independent_mean,n,variance_ratio
0,0,1,2.682940e-06,2.679663e-06,0.302676,0.302676,20,1.001223
1,1,5,4.677397e-08,4.671723e-08,-0.076323,-0.076325,20,1.001215


## 3. External paired statistics (runs only if audited input exists)


In [4]:
metrics_path=RESULTS/'per_user_metrics.csv'
if RUN_ALL and metrics_path.exists():
    metrics=pd.read_csv(metrics_path)
    required={'user_id','model','hit','ndcg'}
    missing=required-set(metrics.columns)
    if missing: raise ValueError(f'Missing columns: {sorted(missing)}')
    ci,tests=paired_user_statistics(metrics)
    ci.to_csv(RESULTS/'paired_bootstrap_ci.csv',index=False)
    tests.to_csv(RESULTS/'paired_tests_holm.csv',index=False)
    display(ci);display(tests)
else:
    print('Phase D skipped: no audited per_user_metrics.csv')

Phase D skipped: no audited per_user_metrics.csv


## 4. Completion manifest


In [5]:
manifest={'run_all':RUN_ALL,'phase_a':'archived_not_rerun','phase_bc':'archived_inspected','crn':'executed_click_feedback_variant','phase_d':'executed' if metrics_path.exists() else 'skipped_missing_per_user_metrics','scalability':'skipped_no_distinct_8_10_player_library','registered_config_changed':False,'crn_click_feedback_weight':CRN_CLICK_FEEDBACK_WEIGHT}
p=RESULTS/'run_all_remaining_manifest.json'
p.write_text(json.dumps(manifest,indent=2))
print(p)
print(json.dumps(manifest,indent=2))

/Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/results/reviewer_phase_assets/run_all_remaining_manifest.json
{
  "run_all": true,
  "phase_a": "archived_not_rerun",
  "phase_bc": "archived_inspected",
  "crn": "executed_click_feedback_variant",
  "phase_d": "skipped_missing_per_user_metrics",
  "scalability": "skipped_no_distinct_8_10_player_library",
  "registered_config_changed": false,
  "crn_click_feedback_weight": 0.35
}


In [6]:
# Bootstrap confidence interval for the variance ratio (20 seeds)
if RUN_ALL and 'crn_summary' in globals():
    rng=np.random.default_rng(20260815)
    ci_rows=[]
    for row in crn_summary.itertuples(index=False):
        subset=crn_rows[(crn_rows.mask_a==row.mask_a)&(crn_rows.mask_b==row.mask_b)]
        ratios=[]
        for _ in range(2000):
            sample=subset.sample(n=len(subset),replace=True,random_state=int(rng.integers(0,2**31-1)))
            den=float(sample.independent_difference.var(ddof=1))
            ratios.append(float(sample.crn_difference.var(ddof=1)/den) if den>0 else np.nan)
        ci_rows.append({'mask_a':row.mask_a,'mask_b':row.mask_b,'variance_ratio':row.variance_ratio,'variance_ratio_ci_low':float(np.nanquantile(ratios,.025)),'variance_ratio_ci_high':float(np.nanquantile(ratios,.975)),'n_seeds':len(subset)})
    crn_ci=pd.DataFrame(ci_rows)
    crn_ci.to_csv(RESULTS/'crn_click_feedback'/'crn_variance_ratio_ci.csv',index=False)
    display(crn_ci)


,mask_a,mask_b,variance_ratio,variance_ratio_ci_low,variance_ratio_ci_high,n_seeds
0,0,1,1.001223,0.997634,1.006012,20
1,1,5,1.001215,0.978161,1.035383,20
